LLM A Hands on appraoch project Gentut

In [1]:
!nvidia-smi

Sat Aug  8 15:20:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [65]:
import os
from google.colab import userdata
os.makedirs('/content/drive/MyDrive/GenTut', exist_ok=True)
os.environ['HF_HOME'] = '/content/drive/MyDrive/GenTut/hf_cache'  # cache model weights across sessions
os.environ['GOOGLE_API_KEY'] = userdata.get('GEMINI_API_KEY')
os.environ['NGROK_TOKEN'] = userdata.get('NGROK_TOKEN')

In [4]:
!git config --global user.email "dev.skg@gmail.com"
!git config --global user.name "Sandesh"
%cd /content/drive/MyDrive/GenTut
!git pull
%cd gentut

/content/drive/MyDrive/GenTut
fatal: destination path 'gentut' already exists and is not an empty directory.
/content/drive/MyDrive/GenTut/gentut


2. Install dependencies

In [6]:
%pip install -q langchain langgraph langchain-huggingface langchain-google-genai \
    transformers accelerate bitsandbytes==0.46.1 pydantic streamlit google-generativeai python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 101.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 90.8 MB/s eta 0:00:00


In [35]:
from dotenv import load_dotenv
import os

load_dotenv('/content/drive/MyDrive/GenTut/.env')

hf_token = os.getenv("HF_TOKEN")
gemini_key = os.getenv("GEMINI_API_KEY")

In [36]:
from huggingface_hub import login
login(token=hf_token)

import google.generativeai as genai
genai.configure(api_key=gemini_key)

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [10]:
messages = [{"role": "user", "content": "Say hello in one sentence."}]
inputs = tokenizer.apply_chat_template(messages,
                                    return_tensors="pt",
                                    add_generation_prompt=True,
                                    return_dict=True).to(model.device)

output = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:447: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


user

Say hello in one sentence.assistant

Hello!


In [16]:
from schemas import CognitiveState, TutorAction
from agents import skill_identifier_agent, profiler_agent
from eval_utils import ResultsLogger

logger = ResultsLogger()

In [15]:
import sys
for mod in ["schemas", "agents", "eval_utils"]:
    if mod in sys.modules:
        del sys.modules[mod]

from schemas import CognitiveState, TutorAction
from agents import skill_identifier_agent, profiler_agent
from eval_utils import ResultsLogger

logger = ResultsLogger()

In [21]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   __pycache__/eval_utils.cpython-312.pyc
	modified:   __pycache__/schemas.cpython-312.pyc
	modified:   agents.py

no changes added to commit (use "git add" and/or "git commit -a")


In [22]:

!git add .
!git commit -m "Step - updated"
!git push

[main ca768bb] Step - updated
 3 files changed, 49 insertions(+), 6 deletions(-)
 rewrite __pycache__/schemas.cpython-312.pyc (68%)
Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 2.15 KiB | 315.00 KiB/s, done.
Total 6 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/sandeshkg/gentut.git
   d472c93..ca768bb  main -> main


In [18]:
from agents import skill_identifier_agent
from eval_utils import ResultsLogger

logger = ResultsLogger()

def test_skill_identifier(student_message):
    state, raw = skill_identifier_agent(student_message, model, tokenizer)
    if state is not None:
        logger.log("skill_identifier", "", student_message, raw, parsed_state=state)
    else:
        logger.log("skill_identifier", "", student_message, raw, error="validation_failed")

In [19]:
def test_profiler_chain(student_message):
    state, raw1 = skill_identifier_agent(student_message, model, tokenizer)
    if state is None:
        print(f"Skill Identifier failed. Raw: {raw1}")
        return
    action, raw2 = profiler_agent(state, model, tokenizer)
    if action is not None:
        logger.log("profiler", "", student_message, raw2, parsed_state=action)
        print(f"State: {state}\nAction: {action}\n")
    else:
        logger.log("profiler", "", student_message, raw2, error="validation_failed")
        print(f"State: {state}\nProfiler failed. Raw: {raw2}\n")

test_profiler_chain("I don't understand why my for loop never terminates.")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:447: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


State: student_id='s001' current_topic='loops' skill_level='novice' misconceptions=['confuses loop termination conditions'] hint_stage=1 mastery_score=0.4
Action: next_action='give_hint' reasoning='Student is still struggling with loop termination conditions and has not yet mastered the topic; a targeted hint can help resolve the misconception.' target_misconception='confuses loop termination conditions'



In [20]:
test_messages = [
    "What's the difference between a list and a tuple?",
    "My recursive function keeps hitting max recursion depth.",
    "I think I understand for loops now, can I try something harder?",
]

for msg in test_messages:
    test_profiler_chain(msg)

State: student_id='s001' current_topic='lists and tuples' skill_level='novice' misconceptions=['may not understand the concept of mutable vs immutable data structures'] hint_stage=0 mastery_score=0.5
Action: next_action='ask_clarifying_question' reasoning='Student is still struggling with the concept and has not fully mastered it, so asking a clarifying question can help identify the source of the issue.' target_misconception=None

State: student_id='s001' current_topic='recursion' skill_level='developing' misconceptions=['lacks understanding of stack overflow and max recursion depth'] hint_stage=2 mastery_score=0.4
Action: next_action='give_hint' reasoning='Student is still struggling with recursion concept, hint stage is 2, and mastery score is low. Providing a targeted hint can help clarify the concept.' target_misconception='lacks understanding of stack overflow and max recursion depth'

State: student_id='s001' current_topic='loops' skill_level='developing' misconceptions=['may no

In [42]:
logger.save("schema_fidelity_log.csv")
!git add .
!git commit -m "Step 3 complete: Content Creator agent (Gemini) built and chained end-to-end with Skill Identifier + Profiler"
!git push

Saved 7 results to schema_fidelity_log.csv
[main 0845ced] Step 3 complete: Content Creator agent (Gemini) built and chained end-to-end with Skill Identifier + Profiler
 1 file changed, 12 insertions(+)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 840 bytes | 210.00 KiB/s, done.
Total 3 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/sandeshkg/gentut.git
   edd445e..0845ced  main -> main


In [25]:
%%writefile -a schemas.py

class TutorContent(BaseModel):
    content_type: Literal["hint", "clarifying_question", "new_content", "mastery_message"]
    message: str = Field(description="The actual text shown to the student")
    difficulty_note: Optional[str] = Field(default=None, description="Internal note on pacing/difficulty, not shown to student")

Appending to schemas.py


In [26]:
%%writefile -a agents.py

from schemas import TutorContent
import google.generativeai as genai

def build_content_creator_prompt(state: CognitiveState, action: TutorAction) -> str:
    example = TutorContent(
        content_type="hint",
        message="Take a look at your loop's condition — does it ever change inside the loop body? If not, that's why it never ends.",
        difficulty_note="Keep hint conceptual, avoid giving the full answer directly."
    ).model_dump_json(indent=2)

    return f"""You are a Content Creator agent in a tutoring system. Generate the actual tutoring content to show the student, based on their cognitive state and the chosen tutor action.

Output ONLY a raw JSON object. No prose, no markdown code fences, no explanation before or after.

Field rules:
- content_type: MUST match the tutor action's intent, one of: "hint", "clarifying_question", "new_content", "mastery_message"
- message: the actual text shown to the student. Be encouraging, concise, and pedagogically sound — guide the student toward understanding rather than giving away the full answer, unless content_type is "mastery_message".
- difficulty_note: a short internal note on pacing, or null

Example of a correctly formatted output:
{example}

Student's current state:
{state.model_dump_json(indent=2)}

Chosen tutor action:
{action.model_dump_json(indent=2)}

JSON output:"""

def content_creator_agent(state: CognitiveState, action: TutorAction, gemini_model) -> tuple:
    """Returns (content, raw_output). content is None if validation failed."""
    prompt = build_content_creator_prompt(state, action)
    response = gemini_model.generate_content(prompt)
    raw = response.text
    cleaned = extract_json(raw)
    try:
        content = TutorContent.model_validate_json(cleaned)
        return content, raw
    except Exception:
        return None, raw

Appending to agents.py


In [27]:
import sys
for mod in ["schemas", "agents"]:
    if mod in sys.modules:
        del sys.modules[mod]

from schemas import CognitiveState, TutorAction, TutorContent
from agents import skill_identifier_agent, profiler_agent, content_creator_agent

In [38]:
gemini = genai.GenerativeModel("gemini-2.5-flash")

In [39]:
def test_full_chain(student_message):
    state, raw1 = skill_identifier_agent(student_message, model, tokenizer)
    if state is None:
        print(f"Skill Identifier failed. Raw: {raw1}")
        return
    action, raw2 = profiler_agent(state, model, tokenizer)
    if action is None:
        print(f"Profiler failed. Raw: {raw2}")
        return
    content, raw3 = content_creator_agent(state, action, gemini)
    if content is not None:
        logger.log("content_creator", "", student_message, raw3, parsed_state=content)
        print(f"State: {state}\nAction: {action}\nContent: {content}\n")
    else:
        logger.log("content_creator", "", student_message, raw3, error="validation_failed")
        print(f"State: {state}\nAction: {action}\nContent creator failed. Raw: {raw3}\n")

test_full_chain("I don't understand why my for loop never terminates.")

State: student_id='s002' current_topic='loops' skill_level='novice' misconceptions=['confuses loop termination conditions'] hint_stage=2 mastery_score=0.1
Action: next_action='give_hint' reasoning='Student has not yet mastered the topic and is still showing misconceptions; a targeted hint is necessary to guide their understanding.' target_misconception='confuses loop termination conditions'
Content: content_type='hint' message='Remember, a loop will continue to run as long as its condition is `true`. What needs to happen inside your loop for that condition to eventually become `false` and allow the loop to stop?' difficulty_note='Hint focuses on the mechanism of loop termination and condition change for a novice at hint_stage 2.'



In [41]:
test_full_chain("I think I understand for loops now, can I try something harder?")
test_full_chain("What's the difference between a list and a tuple?")

State: student_id='s001' current_topic='control structures' skill_level='developing' misconceptions=['may not fully grasp the concept of loop control'] hint_stage=1 mastery_score=0.6
Action: next_action='give_hint' reasoning='Student is still struggling with the concept of loop control and has not yet fully grasped it; a hint at this stage may help them overcome their misconception.' target_misconception='may not fully grasp the concept of loop control'
Content: content_type='hint' message='When you set up a loop, what condition needs to be met for it to continue, and what makes it eventually stop?' difficulty_note="First stage hint focusing on the conceptual role of a loop's stopping condition."

State: student_id='s001' current_topic='lists_and_tuples' skill_level='developing' misconceptions=['may not understand the concept of immutability'] hint_stage=0 mastery_score=0.5
Action: next_action='ask_clarifying_question' reasoning="Student's mastery score is low and hint stage is 0, indi

In [45]:
import sys
for mod in ["schemas", "agents"]:
    if mod in sys.modules:
        del sys.modules[mod]

from schemas import GraphState, CognitiveState, TutorAction, TutorContent
from agents import build_tutoring_graph

tutoring_graph = build_tutoring_graph()

In [49]:
result = tutoring_graph.invoke(GraphState(student_message="I don't understand why my for loop never terminates."))
print(result["tutor_content"])

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:447: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


content_type='hint' message='For a loop to eventually stop, its condition needs to become `false`. Could you review the condition in your loop and think about how the values it checks are changing?' difficulty_note='First hint for a novice on loop termination; encourage reflection on condition change.'


In [50]:
%%writefile -a agents.py

def should_continue(state: GraphState) -> str:
    """Route back to skill_identifier for another turn, or end after N turns."""
    if state.turn_count >= 3:
        return END
    return "wait_for_input"

def make_tutoring_graph_v2(model, tokenizer, gemini_model, max_turns=3):
    def node_skill_identifier(state: GraphState) -> dict:
        cog_state, raw = skill_identifier_agent(state.student_message, model, tokenizer)
        if cog_state is None:
            cog_state = state.cognitive_state or CognitiveState(student_id="s001", current_topic="unclear")
        return {"cognitive_state": cog_state}

    def node_profiler(state: GraphState) -> dict:
        action, raw = profiler_agent(state.cognitive_state, model, tokenizer)
        if action is None:
            action = TutorAction(next_action="ask_clarifying_question", reasoning="fallback")
        return {"tutor_action": action}

    def node_content_creator(state: GraphState) -> dict:
        content, raw = content_creator_agent(state.cognitive_state, state.tutor_action, gemini_model)
        if content is None:
            content = TutorContent(content_type="clarifying_question", message="Could you tell me more?")
        history = state.conversation_history + [state.student_message, content.message]
        return {"tutor_content": content, "conversation_history": history, "turn_count": state.turn_count + 1}

    graph = StateGraph(GraphState)
    graph.add_node("skill_identifier", node_skill_identifier)
    graph.add_node("profiler", node_profiler)
    graph.add_node("content_creator", node_content_creator)

    graph.set_entry_point("skill_identifier")
    graph.add_edge("skill_identifier", "profiler")
    graph.add_edge("profiler", "content_creator")
    graph.add_edge("content_creator", END)  # each invoke() = one turn; loop handled outside, see below

    return graph.compile()

Appending to agents.py


In [57]:
import sys
for mod in ["schemas", "agents"]:
    if mod in sys.modules:
        del sys.modules[mod]
from schemas import GraphState
from agents import make_tutoring_graph_v2
tutoring_graph = make_tutoring_graph_v2(model, tokenizer, gemini)

def run_turn(state, student_message):
    state.student_message = student_message
    result = tutoring_graph.invoke(state)
    return GraphState(**result)

state = GraphState()
turns = [
    "I don't understand why my for loop never terminates.",
    "Oh I see, so I need to update the variable inside the loop?",
    "Got it, that makes sense now!",
]
for msg in turns:
    state = run_turn(state, msg)
    print(f"Student: {msg}")
    print(f"Tutor: {state.tutor_content.message}")
    print(f"[topic={state.cognitive_state.current_topic}, skill_level={state.cognitive_state.skill_level}, hint_stage={state.cognitive_state.hint_stage}]\n")

Student: I don't understand why my for loop never terminates.
Tutor: Remember, a `for` loop needs to know when to stop. Take a look at the condition that determines if your loop should continue running. How does it ensure the loop eventually ends?
[topic=for loops, skill_level=developing, hint_stage=0]

Student: Oh I see, so I need to update the variable inside the loop?
Tutor: Remember, a `for` loop has three main parts: where it starts, when it stops, and how it changes with each repetition. Thinking about each part can often help clarify how the loop will behave.
[topic=for loops, skill_level=developing, hint_stage=0]

Student: Got it, that makes sense now!
Tutor: You're on the right track! For a proficient student like you, sometimes the best way forward is to take a moment and re-evaluate the core objective or definition related to the problem. What key concept do you think applies here?
[topic=your_topic_name, skill_level=proficient, hint_stage=0]



In [59]:
%%writefile agents.py
import re
from schemas import CognitiveState, TutorAction, TutorContent, GraphState
from langgraph.graph import StateGraph, END

# ---------- shared helpers ----------

def extract_json(raw: str) -> str:
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    return match.group(0) if match else raw

PLACEHOLDER_TOPICS = {"same_topic_as_previous", "your_topic_name", "same as previous", "unchanged", "same"}

# ---------- Skill Identifier ----------

def build_skill_identifier_prompt(student_message: str, prior_state=None, recent_history=None) -> str:
    example = CognitiveState(
        student_id="EXAMPLE_ID", current_topic="EXAMPLE_TOPIC_DO_NOT_COPY", skill_level="proficient",
        misconceptions=["EXAMPLE_MISCONCEPTION_DO_NOT_COPY"], hint_stage=2, mastery_score=0.9
    ).model_dump_json(indent=2)

    context_block = ""
    if prior_state is not None:
        context_block = f"""
Previous cognitive state (this reflects the ACTUAL ongoing session — update it based on the new message; don't reset it unless the topic clearly changed):
{prior_state.model_dump_json(indent=2)}
"""
    if recent_history:
        context_block += f"\nRecent conversation:\n" + "\n".join(recent_history[-4:])

    return f"""You are a Skill Identifier agent. Extract/update the student's cognitive state given the conversation so far.

Output ONLY a raw JSON object. No prose, no markdown code fences.

CRITICAL: The example below shows FORMAT ONLY. Its field values are placeholders — do NOT copy them into your answer.

Field rules:
- student_id: keep the same as previous state if given, else any string
- current_topic: always output the actual topic name as a real string (e.g. "for loops", "recursion"). NEVER output placeholder phrases like "same as previous" or "your_topic_name" — if the topic hasn't changed, repeat the exact same real topic string as before.
- skill_level: one of "novice", "developing", "proficient". If the student expresses understanding/confidence (e.g. "got it", "that makes sense"), increase skill_level.
- misconceptions: carry forward unresolved ones from prior state, remove resolved ones, add new ones actually implied by this message.
- hint_stage: increment by 1 from the previous state's hint_stage if another hint was needed on the same misconception; reset to 0 only on a genuinely new topic.
- mastery_score: float 0.0-1.0, generally increasing across turns on the same topic if the student shows understanding.

Example showing FORMAT ONLY (do not reuse these values):
{example}
{context_block}
Latest student message: "{student_message}"

JSON output:"""

def skill_identifier_agent(student_message: str, model, tokenizer, prior_state=None, recent_history=None):
    """Returns (state, raw_output). state is None if validation failed."""
    prompt = build_skill_identifier_prompt(student_message, prior_state, recent_history)
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True, return_dict=True
    ).to(model.device)
    output = model.generate(**inputs, max_new_tokens=300, max_length=None)
    raw = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    cleaned = extract_json(raw)
    try:
        state = CognitiveState.model_validate_json(cleaned)
        # programmatic guard: fix placeholder topic labels rather than relying purely on prompt wording
        if prior_state is not None and state.current_topic.strip().lower() in PLACEHOLDER_TOPICS:
            state.current_topic = prior_state.current_topic
        return state, raw
    except Exception:
        return None, raw

# ---------- Profiler ----------

def build_profiler_prompt(state: CognitiveState) -> str:
    example = TutorAction(
        next_action="give_hint",
        reasoning="Student has attempted twice and still shows the same misconception; a targeted hint is more useful than new content.",
        target_misconception="confuses base case with recursive case"
    ).model_dump_json(indent=2)

    return f"""You are a Profiler agent in a tutoring system. Given the student's current cognitive state, decide what the tutor should do next.

Output ONLY a raw JSON object. No prose, no markdown code fences, no explanation before or after.

Field rules:
- next_action: MUST be exactly one of: "give_hint", "ask_clarifying_question", "present_new_content", "mark_mastered"
- reasoning: a short string explaining the choice
- target_misconception: a string from the student's misconceptions list, or null if not applicable

Example of a correctly formatted output:
{example}

Current student state:
{state.model_dump_json(indent=2)}

JSON output:"""

def profiler_agent(state: CognitiveState, model, tokenizer):
    """Returns (action, raw_output). action is None if validation failed."""
    prompt = build_profiler_prompt(state)
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True, return_dict=True
    ).to(model.device)
    output = model.generate(**inputs, max_new_tokens=300, max_length=None)
    raw = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    cleaned = extract_json(raw)
    try:
        action = TutorAction.model_validate_json(cleaned)
        return action, raw
    except Exception:
        return None, raw

# ---------- Content Creator ----------

def build_content_creator_prompt(state: CognitiveState, action: TutorAction) -> str:
    example = TutorContent(
        content_type="hint",
        message="Take a look at your loop's condition — does it ever change inside the loop body? If not, that's why it never ends.",
        difficulty_note="Keep hint conceptual, avoid giving the full answer directly."
    ).model_dump_json(indent=2)

    return f"""You are a Content Creator agent in a tutoring system. Generate the actual tutoring content to show the student, based on their cognitive state and the chosen tutor action.

Output ONLY a raw JSON object. No prose, no markdown code fences, no explanation before or after.

Field rules:
- content_type: MUST match the tutor action's intent, one of: "hint", "clarifying_question", "new_content", "mastery_message"
- message: the actual text shown to the student. Be encouraging, concise, and pedagogically sound.
- difficulty_note: a short internal note on pacing, or null

Example of a correctly formatted output:
{example}

Student's current state:
{state.model_dump_json(indent=2)}

Chosen tutor action:
{action.model_dump_json(indent=2)}

JSON output:"""

def content_creator_agent(state: CognitiveState, action: TutorAction, gemini_model):
    """Returns (content, raw_output). content is None if validation failed."""
    prompt = build_content_creator_prompt(state, action)
    response = gemini_model.generate_content(prompt)
    raw = response.text
    cleaned = extract_json(raw)
    try:
        content = TutorContent.model_validate_json(cleaned)
        return content, raw
    except Exception:
        return None, raw

# ---------- LangGraph wiring (single source of truth) ----------

def make_tutoring_graph(model, tokenizer, gemini_model):
    def node_skill_identifier(state: GraphState) -> dict:
        cog_state, raw = skill_identifier_agent(
            state.student_message, model, tokenizer,
            prior_state=state.cognitive_state,
            recent_history=state.conversation_history
        )
        if cog_state is None:
            cog_state = state.cognitive_state or CognitiveState(student_id="s001", current_topic="unclear")
        return {"cognitive_state": cog_state}

    def node_profiler(state: GraphState) -> dict:
        action, raw = profiler_agent(state.cognitive_state, model, tokenizer)
        if action is None:
            action = TutorAction(next_action="ask_clarifying_question", reasoning="fallback: profiler validation failed")
        return {"tutor_action": action}

    def node_content_creator(state: GraphState) -> dict:
        content, raw = content_creator_agent(state.cognitive_state, state.tutor_action, gemini_model)
        if content is None:
            content = TutorContent(content_type="clarifying_question", message="Could you tell me more about what you're finding tricky?")
        history = state.conversation_history + [state.student_message, content.message]
        return {"tutor_content": content, "conversation_history": history, "turn_count": state.turn_count + 1}

    graph = StateGraph(GraphState)
    graph.add_node("skill_identifier", node_skill_identifier)
    graph.add_node("profiler", node_profiler)
    graph.add_node("content_creator", node_content_creator)

    graph.set_entry_point("skill_identifier")
    graph.add_edge("skill_identifier", "profiler")
    graph.add_edge("profiler", "content_creator")
    graph.add_edge("content_creator", END)  # each invoke() = one turn; multi-turn loop is driven externally

    return graph.compile()

Overwriting agents.py


In [60]:
import sys
for mod in ["schemas", "agents"]:
    if mod in sys.modules:
        del sys.modules[mod]

from schemas import GraphState
from agents import make_tutoring_graph

tutoring_graph = make_tutoring_graph(model, tokenizer, gemini)

def run_turn(state, student_message):
    state.student_message = student_message
    result = tutoring_graph.invoke(state)
    return GraphState(**result)

state = GraphState()
turns = [
    "I don't understand why my for loop never terminates.",
    "Oh I see, so I need to update the variable inside the loop?",
    "Got it, that makes sense now!",
]
for msg in turns:
    state = run_turn(state, msg)
    print(f"Student: {msg}")
    print(f"Tutor: {state.tutor_content.message}")
    print(f"[topic={state.cognitive_state.current_topic}, skill_level={state.cognitive_state.skill_level}, hint_stage={state.cognitive_state.hint_stage}]\n")

Student: I don't understand why my for loop never terminates.
Tutor: When you write a `for` loop, what part of its structure tells the loop when to stop running?
[topic=for loops, skill_level=developing, hint_stage=1]

Student: Oh I see, so I need to update the variable inside the loop?
Tutor: When does a 'for' loop know it's time to stop? Think about what determines its end condition.
[topic=for loops, skill_level=proficient, hint_stage=1]

Student: Got it, that makes sense now!
Tutor: You're almost there! For `for` loops, it's often helpful to double-check the initial value, the condition, and how the counter changes in each iteration. Is everything set up exactly as you intend?
[topic=for loops, skill_level=proficient, hint_stage=0]



In [61]:
logger.save("schema_fidelity_log.csv")
!git add .
!git commit -m "Step 4 complete: consolidated agents.py into single source of truth, fixed context-injection wiring bug, added placeholder-topic guard, validated clean 3-turn run"
!git push

Saved 7 results to schema_fidelity_log.csv
[main f4dab89] Step 4 complete: consolidated agents.py into single source of truth, fixed context-injection wiring bug, added placeholder-topic guard, validated clean 3-turn run
 3 files changed, 90 insertions(+), 26 deletions(-)
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 3.20 KiB | 654.00 KiB/s, done.
Total 6 (delta 5), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (5/5), completed with 5 local objects.
To https://github.com/sandeshkg/gentut.git
   0845ced..f4dab89  main -> main


In [64]:
!pip install -q pyngrok

In [66]:
from pyngrok import ngrok

# get a free authtoken from ngrok.com/signup if you haven't already
ngrok.set_auth_token(os.getenv("NGROK_TOKEN"))  # add NGROK_TOKEN to your .env, same pattern as HF/Gemini keys

public_url = ngrok.connect(8501)
print(public_url)

!streamlit run app.py &>/content/logs.txt &

NgrokTunnel: "https://catlike-overprice-writing.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
!git add .
!git commit -m "Step 5: Streamlit ui"
!git push